# KL Divergence & ELBO: Interactive Exploration

This notebook provides a hands-on exploration of **Kullback-Leibler (KL) divergence** and the
**Evidence Lower Bound (ELBO)** -- two fundamental concepts behind Variational Autoencoders (VAEs)
and diffusion models.

We will:
1. Compute KL divergence between Gaussians analytically and via Monte Carlo
2. Demonstrate the asymmetry of KL divergence
3. Visualize how KL changes with distribution parameters
4. Compare forward vs reverse KL on a bimodal target (mode-covering vs mode-seeking)
5. Show Monte Carlo KL estimation convergence
6. Explore the ELBO on a toy latent variable model
7. Decompose the ELBO into reconstruction and regularization terms

**Runtime:** Google Colab with L4 GPU recommended.

In [ ]:
# ============================================================
# Colab Setup
# ============================================================
!pip install -q torch matplotlib numpy scipy

import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy import optimize
from scipy.stats import norm

%matplotlib inline

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Device detection
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Plot style
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 13,
    "axes.grid": True,
    "grid.alpha": 0.3,
})
print("Setup complete.")

## 1. KL Divergence Between Gaussians

For two univariate Gaussians $p = \mathcal{N}(\mu_1, \sigma_1^2)$ and $q = \mathcal{N}(\mu_2, \sigma_2^2)$,
the KL divergence has a closed-form expression:

$$
D_{\mathrm{KL}}\!\left(\mathcal{N}(\mu_1, \sigma_1^2) \;\|\; \mathcal{N}(\mu_2, \sigma_2^2)\right)
= \ln\!\frac{\sigma_2}{\sigma_1}
  + \frac{\sigma_1^2 + (\mu_1 - \mu_2)^2}{2\,\sigma_2^2}
  - \frac{1}{2}
$$

Key properties:
- $D_{\mathrm{KL}} \geq 0$ always (Gibbs' inequality)
- $D_{\mathrm{KL}} = 0$ if and only if $p = q$
- It is **not** symmetric: $D_{\mathrm{KL}}(p \| q) \neq D_{\mathrm{KL}}(q \| p)$ in general

In [ ]:
# ============================================================
# Analytical vs Monte Carlo KL Divergence
# ============================================================

def kl_gaussian_analytical(mu1, sigma1, mu2, sigma2):
    """Closed-form KL(N(mu1,sigma1^2) || N(mu2,sigma2^2))."""
    return (np.log(sigma2 / sigma1)
            + (sigma1**2 + (mu1 - mu2)**2) / (2 * sigma2**2)
            - 0.5)


def kl_gaussian_mc(mu1, sigma1, mu2, sigma2, n_samples=500_000):
    """Monte Carlo estimate: E_p[log p(x) - log q(x)]."""
    x = np.random.normal(mu1, sigma1, size=n_samples)
    log_p = norm.logpdf(x, mu1, sigma1)
    log_q = norm.logpdf(x, mu2, sigma2)
    return np.mean(log_p - log_q)


# Example distributions
mu1, sigma1 = 0.0, 1.0   # p = N(0, 1)
mu2, sigma2 = 1.5, 2.0   # q = N(1.5, 4)

kl_analytic = kl_gaussian_analytical(mu1, sigma1, mu2, sigma2)
kl_mc = kl_gaussian_mc(mu1, sigma1, mu2, sigma2)

print(f"p = N({mu1}, {sigma1}^2),  q = N({mu2}, {sigma2}^2)")
print(f"")
print(f"Analytical KL(p || q) = {kl_analytic:.6f}")
print(f"Monte Carlo KL(p || q) = {kl_mc:.6f}")
print(f"Absolute difference   = {abs(kl_analytic - kl_mc):.6f}")
print(f"")
print("The analytical and Monte Carlo estimates match closely.")

## 2. Asymmetry of KL Divergence

KL divergence is **not** a true distance metric because it is **asymmetric**:

$$D_{\mathrm{KL}}(p \| q) \neq D_{\mathrm{KL}}(q \| p)$$

This asymmetry has deep practical consequences:
- **Forward KL** $D_{\mathrm{KL}}(p \| q)$: penalizes $q$ wherever $p$ has mass but $q$ does not (mode-covering)
- **Reverse KL** $D_{\mathrm{KL}}(q \| p)$: penalizes $q$ wherever $q$ has mass but $p$ does not (mode-seeking)

In [ ]:
# ============================================================
# Asymmetry of KL Divergence
# ============================================================

p_mu, p_sigma = 0.0, 1.0    # p = N(0, 1)
q_mu, q_sigma = 2.0, 0.5    # q = N(2, 0.25)

kl_pq = kl_gaussian_analytical(p_mu, p_sigma, q_mu, q_sigma)
kl_qp = kl_gaussian_analytical(q_mu, q_sigma, p_mu, p_sigma)

print(f"p = N({p_mu}, {p_sigma}^2)")
print(f"q = N({q_mu}, {q_sigma}^2)")
print(f"")
print(f"D_KL(p || q) = {kl_pq:.4f}")
print(f"D_KL(q || p) = {kl_qp:.4f}")
print(f"")
print(f"They are NOT equal!  Ratio: {kl_pq / kl_qp:.2f}x")

# Visualization
x = np.linspace(-5, 6, 500)
p_pdf = norm.pdf(x, p_mu, p_sigma)
q_pdf = norm.pdf(x, q_mu, q_sigma)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(x, p_pdf, "b-", linewidth=2.5, label=f"p = N({p_mu}, {p_sigma}$^2$)")
ax.plot(x, q_pdf, "r-", linewidth=2.5, label=f"q = N({q_mu}, {q_sigma}$^2$)")
ax.fill_between(x, p_pdf, alpha=0.15, color="blue")
ax.fill_between(x, q_pdf, alpha=0.15, color="red")

ax.annotate(f"$D_{{KL}}(p \\| q) = {kl_pq:.3f}$",
            xy=(0.05, 0.92), xycoords="axes fraction",
            fontsize=14, color="blue",
            bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.8))
ax.annotate(f"$D_{{KL}}(q \\| p) = {kl_qp:.3f}$",
            xy=(0.05, 0.80), xycoords="axes fraction",
            fontsize=14, color="red",
            bbox=dict(boxstyle="round,pad=0.3", fc="lightyellow", alpha=0.8))

ax.set_xlabel("x")
ax.set_ylabel("Density")
ax.set_title("Asymmetry of KL Divergence")
ax.legend(fontsize=13)
plt.tight_layout()
plt.show()

## 3. KL Divergence Parameter Sweeps

How does $D_{\mathrm{KL}}(p \| q)$ change as we vary the parameters of $q$?

We fix $p = \mathcal{N}(0, 1)$ and sweep:
1. The mean $\mu_q$ while keeping $\sigma_q = 1$
2. The standard deviation $\sigma_q$ while keeping $\mu_q = 0$
3. Both $\mu_q$ and $\sigma_q$ simultaneously (2D heatmap)

In [ ]:
# ============================================================
# KL Divergence Parameter Sweeps
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Subplot 1: Sweep mu_q ---
mu_q_range = np.linspace(-5, 5, 200)
kl_vs_mu = [kl_gaussian_analytical(0, 1, mu_q, 1) for mu_q in mu_q_range]

axes[0].plot(mu_q_range, kl_vs_mu, "b-", linewidth=2)
axes[0].set_xlabel(r"$\mu_q$")
axes[0].set_ylabel(r"$D_{KL}(p \| q)$")
axes[0].set_title(r"Sweep $\mu_q$ (fix $\sigma_q=1$)")
axes[0].axvline(0, color="gray", linestyle="--", alpha=0.5, label=r"$\mu_p=0$")
axes[0].legend()

# --- Subplot 2: Sweep sigma_q ---
sigma_q_range = np.linspace(0.1, 5, 200)
kl_vs_sigma = [kl_gaussian_analytical(0, 1, 0, sq) for sq in sigma_q_range]

axes[1].plot(sigma_q_range, kl_vs_sigma, "r-", linewidth=2)
axes[1].set_xlabel(r"$\sigma_q$")
axes[1].set_ylabel(r"$D_{KL}(p \| q)$")
axes[1].set_title(r"Sweep $\sigma_q$ (fix $\mu_q=0$)")
axes[1].axvline(1, color="gray", linestyle="--", alpha=0.5, label=r"$\sigma_p=1$")
axes[1].legend()

# --- Subplot 3: 2D Heatmap ---
mu_grid = np.linspace(-4, 4, 100)
sigma_grid = np.linspace(0.2, 4, 100)
MU, SIGMA = np.meshgrid(mu_grid, sigma_grid)
KL_2D = np.zeros_like(MU)

for i in range(MU.shape[0]):
    for j in range(MU.shape[1]):
        KL_2D[i, j] = kl_gaussian_analytical(0, 1, MU[i, j], SIGMA[i, j])

# Clip for better visualization
KL_2D_clipped = np.clip(KL_2D, 0, 10)

im = axes[2].pcolormesh(MU, SIGMA, KL_2D_clipped, cmap="viridis", shading="auto")
axes[2].plot(0, 1, "r*", markersize=15, label="p = N(0,1)")
axes[2].set_xlabel(r"$\mu_q$")
axes[2].set_ylabel(r"$\sigma_q$")
axes[2].set_title(r"$D_{KL}(p \| q)$ Heatmap")
axes[2].legend(fontsize=11)
fig.colorbar(im, ax=axes[2], label=r"$D_{KL}$ (clipped at 10)")

plt.tight_layout()
plt.show()

## 4. Forward vs Reverse KL on Bimodal Target

This is the **key visualization** for understanding why the direction of KL matters.

We define a **bimodal** target distribution:

$$p(x) = 0.5 \cdot \mathcal{N}(-3,\, 0.7^2) + 0.5 \cdot \mathcal{N}(3,\, 0.7^2)$$

and try to approximate it with a **single Gaussian** $q(x) = \mathcal{N}(\mu, \sigma^2)$.

**Forward KL** $D_{\mathrm{KL}}(p \| q)$:
- Minimizing this forces $q$ to cover **all** regions where $p$ has mass
- Result: $q$ spreads out to cover both modes (**mode-covering** behavior)
- $\mu \approx 0$, large $\sigma$

**Reverse KL** $D_{\mathrm{KL}}(q \| p)$:
- Minimizing this forces $q$ to avoid placing mass where $p$ has **no** mass
- Result: $q$ collapses onto a **single** mode (**mode-seeking** behavior)
- $\mu \approx \pm 3$, small $\sigma$

This distinction is crucial for understanding VAEs (which minimize reverse KL in the ELBO)
and why they sometimes produce blurry outputs.

In [ ]:
# ============================================================
# FLAGSHIP: Forward vs Reverse KL on Bimodal Target
# ============================================================

def bimodal_pdf(x, mu1=-3, mu2=3, sigma=0.7):
    """PDF of a 50-50 mixture of two Gaussians."""
    return 0.5 * norm.pdf(x, mu1, sigma) + 0.5 * norm.pdf(x, mu2, sigma)


def bimodal_logpdf(x, mu1=-3, mu2=3, sigma=0.7):
    """Log-PDF of bimodal Gaussian mixture (numerically stable)."""
    log_comp1 = norm.logpdf(x, mu1, sigma) + np.log(0.5)
    log_comp2 = norm.logpdf(x, mu2, sigma) + np.log(0.5)
    return np.logaddexp(log_comp1, log_comp2)


def sample_bimodal(n, mu1=-3, mu2=3, sigma=0.7):
    """Sample from the bimodal mixture."""
    components = np.random.choice([0, 1], size=n)
    samples = np.where(
        components == 0,
        np.random.normal(mu1, sigma, size=n),
        np.random.normal(mu2, sigma, size=n),
    )
    return samples


# ---- (a) Forward KL: minimize D_KL(p || q) ----
# D_KL(p||q) = E_p[log p(x)] - E_p[log q(x)]
# Minimizing w.r.t. q means maximizing E_p[log q(x)]
# i.e., maximize the expected log-likelihood of q under samples from p.

def forward_kl_loss(params, p_samples):
    """Negative E_p[log q(x)] -- minimize to get forward KL optimum."""
    mu, log_sigma = params
    sigma = np.exp(log_sigma)
    log_q = norm.logpdf(p_samples, mu, sigma)
    return -np.mean(log_q)


np.random.seed(SEED)
p_samples = sample_bimodal(100_000)

result_fwd = optimize.minimize(
    forward_kl_loss, x0=[0.0, 0.0], args=(p_samples,),
    method="Nelder-Mead",
    options={"maxiter": 5000, "xatol": 1e-8, "fatol": 1e-8},
)
fwd_mu = result_fwd.x[0]
fwd_sigma = np.exp(result_fwd.x[1])
print(f"Forward KL optimum:  mu = {fwd_mu:.4f},  sigma = {fwd_sigma:.4f}")
print(f"  -> Mode-covering: q spreads to cover both modes\n")


# ---- (b) Reverse KL: minimize D_KL(q || p) ----
# D_KL(q||p) = E_q[log q(x) - log p(x)]
# We sample from q and minimize the MC estimate.

def reverse_kl_loss(params):
    """Monte Carlo estimate of D_KL(q || p)."""
    mu, log_sigma = params
    sigma = np.exp(log_sigma)
    # Sample from q
    q_samples = np.random.normal(mu, sigma, size=10_000)
    log_q = norm.logpdf(q_samples, mu, sigma)
    log_p = bimodal_logpdf(q_samples)
    return np.mean(log_q - log_p)


# Run from multiple initializations and keep the best
best_rev_loss = np.inf
best_rev_params = None
np.random.seed(SEED)

for init_mu in [-4.0, -1.0, 0.0, 1.0, 4.0]:
    result_rev = optimize.minimize(
        reverse_kl_loss, x0=[init_mu, -0.3],
        method="Nelder-Mead",
        options={"maxiter": 5000, "xatol": 1e-8, "fatol": 1e-8},
    )
    if result_rev.fun < best_rev_loss:
        best_rev_loss = result_rev.fun
        best_rev_params = result_rev.x

rev_mu = best_rev_params[0]
rev_sigma = np.exp(best_rev_params[1])
print(f"Reverse KL optimum: mu = {rev_mu:.4f},  sigma = {rev_sigma:.4f}")
print(f"  -> Mode-seeking: q collapses onto one mode\n")


# ---- Plot all three curves ----
x = np.linspace(-8, 8, 1000)
p_true = bimodal_pdf(x)
q_fwd = norm.pdf(x, fwd_mu, fwd_sigma)
q_rev = norm.pdf(x, rev_mu, rev_sigma)

fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(x, p_true, "k-", linewidth=3, label="True bimodal p(x)")
ax.plot(x, q_fwd, "b--", linewidth=2.5,
        label=f"Forward KL  q=N({fwd_mu:.1f}, {fwd_sigma:.1f}$^2$) [mode-covering]")
ax.plot(x, q_rev, "r-.", linewidth=2.5,
        label=f"Reverse KL  q=N({rev_mu:.1f}, {rev_sigma:.1f}$^2$) [mode-seeking]")

ax.fill_between(x, p_true, alpha=0.1, color="black")
ax.fill_between(x, q_fwd, alpha=0.08, color="blue")
ax.fill_between(x, q_rev, alpha=0.08, color="red")

ax.set_xlabel("x", fontsize=14)
ax.set_ylabel("Density", fontsize=14)
ax.set_title("Forward vs Reverse KL on Bimodal Target", fontsize=16)
ax.legend(fontsize=12, loc="upper right")
plt.tight_layout()
plt.show()

print("\nKey insight:")
print("  Forward KL (blue) => mode-COVERING: q spreads to cover both modes")
print("  Reverse KL (red)  => mode-SEEKING:  q locks onto a single mode")

## 5. Monte Carlo KL Estimation

In practice we rarely have closed-form KL. Instead we use **Monte Carlo estimates**:

$$D_{\mathrm{KL}}(p \| q) = \mathbb{E}_p\!\left[\log \frac{p(x)}{q(x)}\right] \approx \frac{1}{N}\sum_{i=1}^{N} \left[\log p(x_i) - \log q(x_i)\right], \quad x_i \sim p$$

How many samples do we need for a good estimate?

In [ ]:
# ============================================================
# Monte Carlo KL Estimation -- Convergence
# ============================================================

mc_p_mu, mc_p_sigma = 0.0, 1.0
mc_q_mu, mc_q_sigma = 1.0, 1.5

kl_true = kl_gaussian_analytical(mc_p_mu, mc_p_sigma, mc_q_mu, mc_q_sigma)
print(f"True KL(p||q) = {kl_true:.6f}")
print(f"p = N({mc_p_mu}, {mc_p_sigma}^2),  q = N({mc_q_mu}, {mc_q_sigma}^2)\n")

sample_sizes = [10, 50, 100, 500, 1000, 5000, 10000]
n_runs = 50  # repeated trials for error bars

mc_means = []
mc_stds = []

np.random.seed(SEED)
for N in sample_sizes:
    estimates = []
    for _ in range(n_runs):
        x = np.random.normal(mc_p_mu, mc_p_sigma, size=N)
        log_p = norm.logpdf(x, mc_p_mu, mc_p_sigma)
        log_q = norm.logpdf(x, mc_q_mu, mc_q_sigma)
        estimates.append(np.mean(log_p - log_q))
    mc_means.append(np.mean(estimates))
    mc_stds.append(np.std(estimates))

mc_means = np.array(mc_means)
mc_stds = np.array(mc_stds)

fig, ax = plt.subplots(figsize=(10, 6))

ax.axhline(kl_true, color="green", linewidth=2, linestyle="--",
           label=f"True KL = {kl_true:.4f}")
ax.errorbar(sample_sizes, mc_means, yerr=2 * mc_stds,
            fmt="o-", color="blue", linewidth=2, markersize=8, capsize=5,
            label=r"MC estimate $\pm$ 2 std")

ax.set_xscale("log")
ax.set_xlabel("Number of samples (N)", fontsize=13)
ax.set_ylabel(r"$\hat{D}_{KL}$", fontsize=13)
ax.set_title("Monte Carlo KL Estimation Convergence", fontsize=15)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print("\nAs N increases, the MC estimate converges to the true value")
print("and the variance shrinks as O(1/N).")

## 6. The Evidence Lower Bound (ELBO)

Consider a **latent variable model** with:
- Prior: $p(z) = \mathcal{N}(0, 1)$
- Likelihood: $p(x \mid z) = \mathcal{N}(z,\, 0.5^2)$

The marginal likelihood (evidence) is:

$$p(x) = \int p(x \mid z)\, p(z)\, dz$$

For this model, $p(x)$ is tractable (it is a Gaussian), but in general it is intractable.

The **ELBO** provides a lower bound on $\log p(x)$:

$$\log p(x) \geq \underbrace{\mathbb{E}_{q(z|x)}\!\left[\log p(x|z)\right]}_{\text{reconstruction}} - \underbrace{D_{\mathrm{KL}}\!\left(q(z|x) \| p(z)\right)}_{\text{regularization}} = \mathrm{ELBO}$$

The gap between $\log p(x)$ and the ELBO equals $D_{\mathrm{KL}}(q(z|x) \| p(z|x))$ -- the KL
between our approximate posterior and the true posterior.

In [ ]:
# ============================================================
# ELBO on a Toy Latent Variable Model
# ============================================================

# Model: p(z) = N(0,1),  p(x|z) = N(z, sigma_x^2)
sigma_x = 0.5

# Marginal: p(x) = N(0, 1 + sigma_x^2)  (convolution of two Gaussians)
sigma_marginal = np.sqrt(1.0 + sigma_x**2)

# True posterior: p(z|x) = N(m_true, s_true^2)
# Using Bayes for Gaussians:
#   s_true^2 = 1 / (1/1 + 1/sigma_x^2) = sigma_x^2 / (1 + sigma_x^2)
#   m_true   = s_true^2 * (x / sigma_x^2) = x / (1 + sigma_x^2)

def true_posterior_params(x_obs):
    """True posterior p(z|x) parameters."""
    s2 = sigma_x**2 / (1 + sigma_x**2)
    m = x_obs / (1 + sigma_x**2)
    return m, np.sqrt(s2)


def compute_elbo(x_obs, m, s, n_mc=50000):
    """
    ELBO = E_q[log p(x|z)] - D_KL(q(z|x) || p(z))
    q(z|x) = N(m, s^2)
    """
    # KL(q || prior) analytically
    kl_term = kl_gaussian_analytical(m, s, 0.0, 1.0)
    # E_q[log p(x|z)] via MC
    z_samples = np.random.normal(m, s, size=n_mc)
    log_pxz = norm.logpdf(x_obs, z_samples, sigma_x)
    recon_term = np.mean(log_pxz)
    return recon_term - kl_term, recon_term, -kl_term


# Fix an observation
x_obs = 2.0
log_px_true = norm.logpdf(x_obs, 0.0, sigma_marginal)
m_true, s_true = true_posterior_params(x_obs)

print(f"Observation: x = {x_obs}")
print(f"True log p(x) = {log_px_true:.4f}")
print(f"True posterior: p(z|x) = N({m_true:.4f}, {s_true:.4f}^2)\n")

# Sweep m (variational mean) with fixed s = s_true
m_range = np.linspace(-2, 4, 200)
elbos = []
np.random.seed(SEED)
for m_val in m_range:
    elbo_val, _, _ = compute_elbo(x_obs, m_val, s_true)
    elbos.append(elbo_val)
elbos = np.array(elbos)

# Compute gap = KL(q || true posterior)
gaps = [kl_gaussian_analytical(m_val, s_true, m_true, s_true)
        for m_val in m_range]
gaps = np.array(gaps)

fig, ax = plt.subplots(figsize=(10, 6))

ax.axhline(log_px_true, color="green", linewidth=2.5, linestyle="--",
           label=f"True log p(x) = {log_px_true:.3f}")
ax.plot(m_range, elbos, "b-", linewidth=2.5, label="ELBO(m)")
ax.fill_between(m_range, elbos, log_px_true, alpha=0.15, color="red",
                label=r"Gap = $D_{KL}(q \| p_{\mathrm{true\ post}})$")
ax.axvline(m_true, color="purple", linewidth=1.5, linestyle=":",
           label=f"True posterior mean = {m_true:.3f}")

ax.set_xlabel("Variational mean m", fontsize=13)
ax.set_ylabel("log p(x)  /  ELBO", fontsize=13)
ax.set_title(f"ELBO vs Variational Mean (x={x_obs}, s={s_true:.3f} fixed)", fontsize=15)
ax.legend(fontsize=11, loc="lower left")
ax.set_ylim(log_px_true - 5, log_px_true + 0.5)
plt.tight_layout()
plt.show()

print(f"ELBO is maximized at m = {m_range[np.argmax(elbos)]:.3f} (true posterior mean = {m_true:.3f})")
print(f"At the optimum, the gap closes: ELBO -> log p(x)")

## 7. ELBO Decomposition: Reconstruction vs Regularization

The ELBO has two competing terms:

$$\mathrm{ELBO} = \underbrace{\mathbb{E}_{q(z|x)}[\log p(x|z)]}_{\text{Reconstruction term}} \;-\; \underbrace{D_{\mathrm{KL}}(q(z|x) \| p(z))}_{\text{Regularization (KL) term}}$$

- **Reconstruction term**: Wants $q(z|x)$ to concentrate around $z$ values that explain $x$ well.
  Pushes $q$ toward the data.
- **KL regularization term**: Wants $q(z|x)$ to stay close to the prior $p(z) = \mathcal{N}(0,1)$.
  Pushes $q$ toward the prior.

This creates a **fundamental tradeoff** in VAEs.

In [ ]:
# ============================================================
# ELBO Decomposition: Reconstruction vs Regularization
# ============================================================

m_range_decomp = np.linspace(-2, 4, 300)
recon_terms = []
kl_terms = []
elbo_terms = []

np.random.seed(SEED)
for m_val in m_range_decomp:
    elbo_val, recon_val, neg_kl_val = compute_elbo(x_obs, m_val, s_true)
    recon_terms.append(recon_val)
    kl_terms.append(neg_kl_val)  # already negated
    elbo_terms.append(elbo_val)

recon_terms = np.array(recon_terms)
kl_terms = np.array(kl_terms)
elbo_terms = np.array(elbo_terms)

fig, ax = plt.subplots(figsize=(12, 7))

ax.plot(m_range_decomp, recon_terms, "g-", linewidth=2.5,
        label=r"Reconstruction: $\mathbb{E}_q[\log p(x|z)]$")
ax.plot(m_range_decomp, kl_terms, "r-", linewidth=2.5,
        label=r"Regularization: $-D_{KL}(q \| p(z))$")
ax.plot(m_range_decomp, elbo_terms, "b-", linewidth=3,
        label="ELBO = Reconstruction + Regularization")
ax.axhline(log_px_true, color="black", linewidth=1.5, linestyle="--",
           alpha=0.7, label=f"True log p(x) = {log_px_true:.3f}")

# Mark optima
recon_peak = m_range_decomp[np.argmax(recon_terms)]
kl_peak = m_range_decomp[np.argmax(kl_terms)]
elbo_peak = m_range_decomp[np.argmax(elbo_terms)]

ax.axvline(recon_peak, color="green", linestyle=":", alpha=0.5)
ax.axvline(kl_peak, color="red", linestyle=":", alpha=0.5)
ax.axvline(elbo_peak, color="blue", linestyle=":", alpha=0.5)

ax.annotate(f"Recon peak\nm={recon_peak:.2f}",
            xy=(recon_peak, np.max(recon_terms)),
            xytext=(recon_peak + 0.8, np.max(recon_terms) + 0.3),
            fontsize=10, color="green",
            arrowprops=dict(arrowstyle="->", color="green"))
ax.annotate(f"KL peak\nm={kl_peak:.2f}",
            xy=(kl_peak, np.max(kl_terms)),
            xytext=(kl_peak - 2.0, np.max(kl_terms) + 0.3),
            fontsize=10, color="red",
            arrowprops=dict(arrowstyle="->", color="red"))
ax.annotate(f"ELBO peak\nm={elbo_peak:.2f}",
            xy=(elbo_peak, np.max(elbo_terms)),
            xytext=(elbo_peak + 0.8, np.max(elbo_terms) + 0.5),
            fontsize=10, color="blue",
            arrowprops=dict(arrowstyle="->", color="blue"))

ax.set_xlabel("Variational mean m", fontsize=13)
ax.set_ylabel("Value", fontsize=13)
ax.set_title(f"ELBO Decomposition (x={x_obs}, s={s_true:.3f} fixed)", fontsize=15)
ax.legend(fontsize=11, loc="lower left")
plt.tight_layout()
plt.show()

print("Key observation:")
print(f"  Reconstruction term peaks at m = {recon_peak:.2f} (wants q near the data)")
print(f"  KL regularization peaks at m = {kl_peak:.2f} (wants q near prior mean=0)")
print(f"  ELBO peak (compromise) at m = {elbo_peak:.2f} (= true posterior mean {m_true:.2f})")
print(f"")
print("The ELBO optimum balances reconstruction fidelity with prior regularization.")
print("This is the fundamental tradeoff in VAEs.")

## Key Takeaways

1. **KL divergence is asymmetric.** Forward KL (mode-covering) and reverse KL (mode-seeking) give
   very different results when the target is multimodal.

2. **VAEs optimize the reverse KL** $D_{\mathrm{KL}}(q(z|x) \| p(z|x))$ indirectly through the
   ELBO. This means they tend to be mode-seeking in the posterior, which can lead to underfitting
   complex posteriors.

3. **The ELBO decomposes** into a reconstruction term (fit the data) and a KL regularization term
   (stay close to the prior). The optimal variational posterior balances these two objectives.

4. **The gap** between the ELBO and the true log-evidence equals $D_{\mathrm{KL}}(q \| p_{\text{true posterior}})$.
   A tighter ELBO means a better approximate posterior.

5. **Connection to diffusion models:** Diffusion models can be viewed as hierarchical VAEs with
   a fixed forward process. The training objective (denoising score matching) relates to the ELBO
   decomposed across diffusion timesteps. The KL terms at each step involve Gaussians, just like
   the examples in this notebook.